# Swin Transformer: Hierarchical Vision Transformer using Shifted Windows

## Main Problems of Standard Vision Transformers (ViT)

### 1. Fixed-Scale Tokens (No Hierarchical Representation)

ViT splits the image into fixed-size patches:

$$
\text{Number of patches per dimension}=
\frac{224}{16}=14
$$

Therefore:

$$
14\times14=196 \text{ patches/tokens}
$$

Where:
- $224 \times 224$ = image resolution
- $16 \times 16$ = patch size
- $14 \times 14$ = number of patches
- $196$ = total Transformer tokens

All tokens have the same spatial scale.

This is problematic because objects in images can appear at very different sizes:
- a nearby car may occupy half the image
- a distant car may occupy only a few pixels

Unlike CNNs, ViT does not naturally build hierarchical features such as:

```text
edges → textures → parts → objects
```

This makes standard ViT less effective for:
- object detection
- segmentation
- dense prediction tasks

---

### 2. Quadratic Cost of Global Self-Attention

The computational cost of self-attention grows quadratically with the number of tokens:

$$
O(N^2)
$$

Example:

For a 224×224 image with 16×16 patches:

$$
N=\frac{224\times224}{16\times16}=196
$$

Self-attention cost:

$$
196^2=38416
$$

Still manageable.

But for a 1024×1024 image:

$$
N=\frac{1024\times1024}{16\times16}=4096
$$

Now the attention cost becomes:

$$
4096^2\approx1.68\times10^7
$$

This becomes extremely expensive for:
- high-resolution images
- semantic segmentation
- video transformers

---

### Swin Transformer Main Idea

Swin Transformer solves the two main limitations of standard ViT:

1. lack of hierarchical representations
2. quadratic computational cost of global self-attention


## Arquitecture

Explanation: https://www.youtube.com/watch?v=ORWdELQ1h9M

Example: https://www.youtube.com/watch?v=OX089A-p_Cg

<div>
    <img src='../../images/SwinArquitecture.png' width="800">
</div>

It has many stages and each stage the output is a different dimension tensor. 

### Patch partition

Similar to ViT, we need to partition the RGB image in patches of $4 \times 4$, hence each patch has $4 \times 4 \times 3 = 48$ values, and each dimension has a total patches of $H/4$ and $W/4$, which gives the output shape of the *Patch partition*:

$$
\frac{H}{4} \times \frac{W}{4} \times 48
$$

### Linear embedding

A learnable linear projection that maps image patches into a vector space that transformers can process.

### Swin Transformer Block

<div>
    <img src='../../images/SwinBlock.png' width="250">
</div>

As we can see, the arquitecture is similar to the vanilla transformer, however the key difference is in the **W-MSA (Window Multihead Self Attention)** and **SW-MSA (Shifted Window Multihead Self Attention)**. The vaniila has the MSA and the Masked MSA. This attention strategy is efficient since are linear with respect to the input token size, tackling the quadratic computational cost problem.

**W-MSA**

A window in Swin Transformers, is a collection of patches. So, instead of performing self attention of a patch with respect to all patches, we only do it along the patches inside the window.

<div>
    <img src='../../images/WMSA.png' width="800">
</div>

as it can be noticed, there is no attetion between patches in different windows. The window size is $M = 2$ (patches per dimension).

**SW-MSA**

In order to achived connections between patches of different windows while maintining computational efficancy, the Shifted Windows move de Window by step of $M/2$.

For example, if the windows size is $M=4$, then the window is shifted by 2 to left and 2 to top, this moves the windows as the middle imagaes, and for filling the whole image it is filld with windows of same size, with give a total of 9 windows like the third image.

<div>
    <img src='../../images/Swin.png' width="600">
</div>

but for W-MSA we have only 4 windows, so, we need to pack those 9 windows into 4.

**Cyclic Shift**

Cyclic shift instead of moving the window, it moves the blocks of the image, so the total windows are always 4.

For example: The original image contains 4 sections, so each section is shifted by 2 to the top and 2 to the left, when the section goes out the image it is appended at the botton, then eacth window is passed to the Masked MSA.

<div>
    <img src='../../images/CyclicShift.png' width="800">
</div>

However now another problem arise, now sections are kind of mixed, and there are pixels that does not have relationship, for example, now B is at the right side of D, so in that border pixels does not related to other one, this problem is solved by reversing the cyclic shift.

<div>
    <img src='../../images/RCyclicShift.png' width="500">
</div>


### Patch Merging

Patch Merging is analogous to a pooling layer in CNNs. It reduces the spatial dimensions by a factor of 2 at each stage.

It divides the feature map into non-overlapping 2×2 groups of patches and concatenates the features within each group. Finally, a linear projection is applied to reduce the feature dimension while preserving the most relevant information.

<div>
    <img src='../../images/PatchMerging.png' width="1000">
</div>

### Task Specific Head

Now depending the task, the output would be taking either from the last layer (classification), or for each stage (object detection, segmentation, etc).